In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

# Set HF token for authenticated access (optional but recommended)
# Get your token from https://huggingface.co/settings/tokens
# os.environ["HF_TOKEN"] = "your_token_here"

from datasets import load_dataset

dataset = load_dataset("CShorten/ML-ArXiv-Papers")


Load the ML-ArXiv-Papers dataset from Hugging Face. The `datasets` library uses Apache Arrow for memory-mapped loading, so it handles large datasets efficiently without running out of RAM.

In [2]:
import pandas as pd
df = dataset["train"].to_pandas()
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'], dtype='str')

In [3]:
df

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...,...,...
117587,4995,NaN,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,4996,NaN,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,4997,NaN,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,4998,NaN,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [4]:
df = df[['title', 'abstract']]
df=df.head(50000)
df.isnull().sum()

title       0
abstract    0
dtype: int64

Drop the unnamed index columns and keep only `title` and `abstract`. The index columns are just leftover CSV artifacts with no semantic value. We also limit to 50k rows to keep processing manageable.

Merge title and abstract into a single text block. This gives the sentence transformer more context when generating embeddings.

In [5]:
df["paper_text"] = df["title"] + " " + df["abstract"]
df

,title,abstract,paper_text
0,Learning from compressed observations,The problem of statistical learning is to co...,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun...",Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...,Parametric Learning and Monte Carlo Optimizati...
...,...,...,...
49995,"See, Attend and Brake: An Attention-based Sali...",Visual perception is the most critical input...,"See, Attend and Brake: An Attention-based Sali..."
49996,SNIFF: Reverse Engineering of Neural Networks ...,Neural networks have been shown to be vulner...,SNIFF: Reverse Engineering of Neural Networks ...
49997,Beyond Dropout: Feature Map Distortion to Regu...,Deep neural networks often consist of a grea...,Beyond Dropout: Feature Map Distortion to Regu...
49998,A study of resting-state EEG biomarkers for de...,Background: Depression has become a major he...,A study of resting-state EEG biomarkers for de...


Concatenate title and abstract with a space separator. The space prevents the last word of the title from merging with the first word of the abstract during tokenization.

In [6]:
df['paper_text'].head()

0    Learning from compressed observations   The pr...
1    Sensor Networks with Random Links: Topology De...
2    The on-line shortest path problem under partia...
3    A neural network approach to ordinal regressio...
4    Parametric Learning and Monte Carlo Optimizati...
Name: paper_text, dtype: str

Preview the first few entries of the combined text column.

In [7]:
df[["paper_text"]].head()


,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


Using double brackets `df[["paper_text"]]` returns a DataFrame (2D table) instead of a Series (1D list), which gives better formatting in Jupyter.

In [8]:
print(df['paper_text'].iloc[0])

Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illustrated on the example of nonparametric regress

Load the sentence transformer model. Using `all-MiniLM-L6-v2` because it's lightweight, fast, and outputs 384-dimensional embeddings that work well for semantic search.

In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8500.85it/s]


The `sentence-transformers` library wraps Hugging Face transformers to generate embeddings for entire sentences. `all-MiniLM-L6-v2` maps text to 384-dimensional vectors, which is a good balance between accuracy and speed for our dataset size.

In [10]:
print(type(model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


Grab a single sample to test the encoding pipeline before processing the full dataset.

In [11]:
sample_text = df["paper_text"].iloc[0]
sample_text

'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rate functions. The ideas are\nillustrated on the example of nonparam

`.iloc[0]` extracts the first row by integer position. Testing on a single sample helps catch issues early before running the full encoding.

Remove newline characters and extra whitespace from the text. PDF-extracted text often has arbitrary line breaks that can interfere with tokenization.

In [12]:
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False)
df["paper_text"] = df["paper_text"].str.strip()

`.str.replace("\n", " ")` replaces line breaks with spaces. `regex=False` makes it faster since we're matching literal characters. `.strip()` removes leading/trailing whitespace.

In [13]:
sample_text = df["paper_text"].iloc[0]
sample_text

'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable predictors are drawn from some specified class, and the goal is to approach asymptotically the performance (expected loss) of the best predictor in the class. We consider the setting in which one has perfect observation of the $X$-part of the sample, while the $Y$-part has to be communicated at some finite bit rate. The encoding of the $Y$-values is allowed to depend on the $X$-values. Under suitable regularity conditions on the admissible predictors, the underlying family of probability distributions and the loss function, we give an information-theoretic characterization of achievable predictor performance in terms of conditional distortion-rate functions. The ideas are illustrated on the example of nonparametric regres

Encode the test text through the model. The output is a numpy array of shape (384,) containing the semantic embedding.

In [14]:
embedding = model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


`model.encode()` converts text to a 384-dimensional numpy array. The shape (384,) confirms the model is working correctly.

In [15]:
embedding[:56]

array([-0.13156407, -0.00678268, -0.00367609,  0.03265155,  0.11219645,
        0.01227268,  0.09816722, -0.0900523 ,  0.04231159, -0.01977346,
       -0.03308418,  0.07452943,  0.10632038, -0.02060433, -0.02052102,
        0.00169493,  0.07081953,  0.05854451, -0.11231908,  0.02082476,
        0.05692544,  0.02015779,  0.02583109,  0.03217026,  0.10513762,
       -0.09676762,  0.02700798, -0.02345093, -0.04549677, -0.010137  ,
       -0.01794856, -0.04814427,  0.01077653, -0.03759069,  0.01943479,
        0.03715185,  0.02967846,  0.04330941,  0.04373211,  0.03704869,
       -0.00182598,  0.00455187, -0.00799064,  0.0303737 , -0.01437799,
        0.03795149,  0.0595916 , -0.02583358, -0.06521575,  0.05900267,
       -0.02107136,  0.07359426, -0.05720104,  0.00294057,  0.00767511,
       -0.03334162], dtype=float32)

In [16]:
sample_embedding = model.encode(
    df["paper_text"].head(5).to_list()
)

print(sample_embedding.shape)

(5, 384)


Batch encode 5 papers to verify the output shape. Each paper gets its own 384-dimensional vector, resulting in a (5, 384) matrix.

Load cosine similarity from scikit-learn. Cosine similarity measures the angle between vectors, making it ideal for text comparison since it focuses on semantic direction rather than document length.

In [17]:
import sklearn
from sklearn.metrics.pairwise import cosine_similarity

Import cosine similarity. This function measures the angle between vectors - unlike Euclidean distance, it's not affected by document length, making it ideal for text comparison.

In [18]:
similarity = cosine_similarity([sample_embedding[0]], [sample_embedding[0]])
print(similarity)

[[1.0000001]]


Cosine similarity works on 2D arrays, so we wrap the embedding in brackets to create shape (1, 384). Comparing a vector to itself gives a score of ~1.0.

In [19]:
similarity = cosine_similarity(sample_embedding[0].reshape(1, -1), sample_embedding[1].reshape(1, -1))
print(similarity)

[[0.36625272]]


Using `.reshape(1, -1)` to convert 1D arrays to 2D for cosine similarity. The score of ~0.36 indicates moderate similarity between these two ML papers.

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

for i in range(1, 5):
    sim = cosine_similarity(sample_embedding[0].reshape(1, -1), sample_embedding[i].reshape(1, -1))
    print(f"Similarity with Paper {i}: {sim[0][0]}")

Similarity with Paper 1: 0.36625272035598755
Similarity with Paper 2: 0.33522844314575195
Similarity with Paper 3: 0.1550510972738266
Similarity with Paper 4: 0.37421542406082153


Encode the entire dataset. The model processes papers in batches of 32 to avoid memory issues. This generates a (50000, 384) embedding matrix.

In [21]:
embedding = model.encode(df["paper_text"].to_list(), batch_size=32, show_progress_bar=True)

Batches: 100%|██████████| 1563/1563 [28:36<00:00,  1.10s/it]


`to_list()` converts the DataFrame column to a Python list since sentence-transformers expects that format. `batch_size=32` prevents memory issues by processing papers in chunks.

In [ ]:
import numpy as np
import os
index_path = "data/arxiv_embeddings.npy"

if os.path.exists(index_path):
    print("Loading saved embeddings")
    embeddings = np.load(index_path)
else:
    print("Generating embeddings")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    np.save(index_path, embeddings)
    print("Embeddings saved successfully!")
np.save(index_path, embedding)

print("Embeddings successfully saved to disk!")

Generating embeddings


Batches:   5%|▍         | 71/1563 [01:34<29:20,  1.18s/it]

In [ ]:
df.to_csv("data/cleaned_arxiv_papers.csv", index=False)
print("Cleaned DataFrame saved successfully!")